# SNR Stats (Time-Varying SNR)

This notebook computes time-varying SNR per cell using `compute_time_varying_snr_from_trace`, then reports:
1. How many cells have at least 5 minutes of good-SNR recording (kept cells).
2. For kept cells, the percentage of time deleted due to low SNR.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

current_dir = Path(os.getcwd()).resolve()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.preprocess_neural import compute_time_varying_snr_from_trace


In [2]:
# Parameters
data_root = project_root / 'data'
figures_root = project_root / 'figures'

animals = [
    'CKII_pAce21_PR_20250806',
    'CKII_pAce38_PX_20251126',
    'CKII_pAce45_PX_20260118',
    'CKII_pAce47_PX_20260128',
    'CKII_pAce46_PR_20260222',
]

snr_threshold = 3.5
min_good_minutes = 5.0

# compute_time_varying_snr_from_trace settings
isi_threshold_ms = 20
spike_baseline_points = 3
spike_remove_points = 3
baseline_window_seconds = 10
min_points_per_baseline_window = 20

print('Parameters:')
print(f'  snr_threshold = {snr_threshold}')
print(f'  min_good_minutes = {min_good_minutes}')
print(f'  baseline_window_seconds = {baseline_window_seconds}')


Parameters:
  snr_threshold = 3.5
  min_good_minutes = 5.0
  baseline_window_seconds = 10


In [3]:
def _load_merged_data(animal_dir: Path):
    primary = animal_dir / 'merged_aligned_data.pkl'
    fallback = animal_dir / 'merged_aligned_data_CS.pkl'
    src = primary if primary.exists() else fallback
    if not src.exists():
        raise FileNotFoundError(f'Missing merged data for {animal_dir.name}')
    with src.open('rb') as f:
        return pickle.load(f)


def _safe_cell_array(arr_like, idx, default=None):
    if arr_like is None:
        return default
    if idx < len(arr_like):
        return arr_like[idx]
    return default


In [4]:
rows = []

for animal_id in animals:
    animal_dir = data_root / animal_id
    merged = _load_merged_data(animal_dir)

    frame_rate = float(merged['frame_rate'])
    traces = merged.get('traces_SNR_interpolated', merged.get('traces', []))
    all_spikes = merged.get('all_spikes', merged.get('spikes', []))
    complex_bursts_dicts = merged.get('complex_bursts_dicts', [])

    n_cells = min(len(traces), len(all_spikes))
    print(f'{animal_id}: processing {n_cells} cells')

    for cell_idx in range(n_cells):
        trace = np.asarray(_safe_cell_array(traces, cell_idx, default=[]), dtype=float)
        spks = np.asarray(_safe_cell_array(all_spikes, cell_idx, default=[]), dtype=int)
        complex_burst_dict = _safe_cell_array(complex_bursts_dicts, cell_idx, default=None)

        if trace.ndim != 1 or trace.size == 0:
            rows.append({
                'animal_id': animal_id,
                'cell_idx': int(cell_idx),
                'status': 'skip_empty_trace',
            })
            continue

        try:
            res = compute_time_varying_snr_from_trace(
                trace=trace,
                spks=spks,
                complex_burst_dict=complex_burst_dict,
                sampling_rate_hz=frame_rate,
                isi_threshold_ms=isi_threshold_ms,
                spike_baseline_points=spike_baseline_points,
                spike_remove_points=spike_remove_points,
                baseline_window_seconds=baseline_window_seconds,
                min_points_per_baseline_window=min_points_per_baseline_window,
                plot_single_cell=False,
            )
        except Exception as exc:
            rows.append({
                'animal_id': animal_id,
                'cell_idx': int(cell_idx),
                'status': 'error',
                'error': str(exc),
            })
            continue

        snr_t = np.asarray(res.get('snr_time_varying', []), dtype=float)
        if snr_t.ndim != 1 or snr_t.size == 0:
            rows.append({
                'animal_id': animal_id,
                'cell_idx': int(cell_idx),
                'status': 'skip_empty_snr',
            })
            continue

        total_frames = int(snr_t.size)
        good_mask = np.isfinite(snr_t) & (snr_t >= snr_threshold)
        good_frames = int(np.sum(good_mask))
        deleted_frames = int(total_frames - good_frames)

        total_minutes = total_frames / frame_rate / 60.0
        good_minutes = good_frames / frame_rate / 60.0
        deleted_minutes = deleted_frames / frame_rate / 60.0
        deleted_pct_total = 100.0 * deleted_frames / total_frames if total_frames > 0 else np.nan

        keep_cell = bool(good_minutes >= min_good_minutes)

        rows.append({
            'animal_id': animal_id,
            'cell_idx': int(cell_idx),
            'status': 'ok',
            'frame_rate_hz': frame_rate,
            'total_frames': total_frames,
            'good_frames': good_frames,
            'deleted_frames': deleted_frames,
            'total_minutes': total_minutes,
            'good_minutes': good_minutes,
            'deleted_minutes': deleted_minutes,
            'deleted_pct_total': deleted_pct_total,
            'keep_cell': keep_cell,
        })

cell_stats = pd.DataFrame(rows)
cell_stats.head()


CKII_pAce21_PR_20250806: processing 9 cells
CKII_pAce38_PX_20251126: processing 9 cells
CKII_pAce45_PX_20260118: processing 16 cells
CKII_pAce47_PX_20260128: processing 5 cells
CKII_pAce46_PR_20260222: processing 9 cells


,animal_id,cell_idx,status,frame_rate_hz,total_frames,good_frames,deleted_frames,total_minutes,good_minutes,deleted_minutes,deleted_pct_total,keep_cell
0,CKII_pAce21_PR_20250806,0,ok,500.0,299500,299500,0,9.983333,9.983333,0.0,0.0,True
1,CKII_pAce21_PR_20250806,1,ok,500.0,299500,299500,0,9.983333,9.983333,0.0,0.0,True
2,CKII_pAce21_PR_20250806,2,ok,500.0,299500,299500,0,9.983333,9.983333,0.0,0.0,True
3,CKII_pAce21_PR_20250806,3,ok,500.0,299500,299500,0,9.983333,9.983333,0.0,0.0,True
4,CKII_pAce21_PR_20250806,4,ok,500.0,299500,299500,0,9.983333,9.983333,0.0,0.0,True


In [5]:
valid = cell_stats[cell_stats['status'] == 'ok'].copy()
kept = valid[valid['keep_cell']].copy()

n_total = len(valid)
n_kept = len(kept)
kept_pct = 100.0 * n_kept / n_total if n_total > 0 else np.nan

print('=== Overall ===')
print(f'Total valid cells: {n_total}')
print(f'Kept cells (good SNR >= {min_good_minutes:.1f} min): {n_kept} ({kept_pct:.1f}%)')

if n_kept > 0:
    mean_deleted = float(kept['deleted_pct_total'].mean())
    median_deleted = float(kept['deleted_pct_total'].median())
    weighted_deleted = 100.0 * kept['deleted_frames'].sum() / kept['total_frames'].sum()
    print(f'Kept cells - deleted time (% of recording): mean={mean_deleted:.2f}%, median={median_deleted:.2f}%, weighted={weighted_deleted:.2f}%')
else:
    print('No kept cells found with current threshold/minutes criterion.')

animal_counts = valid.groupby('animal_id', as_index=False).agg(
    total_cells=('cell_idx', 'count'),
    kept_cells=('keep_cell', 'sum'),
)
animal_counts['kept_pct'] = 100.0 * animal_counts['kept_cells'] / animal_counts['total_cells']

if n_kept > 0:
    kept_stats = kept.groupby('animal_id', as_index=False).agg(
        mean_deleted_pct_kept=('deleted_pct_total', 'mean'),
        deleted_frames_kept=('deleted_frames', 'sum'),
        total_frames_kept=('total_frames', 'sum'),
    )
    kept_stats['weighted_deleted_pct_kept'] = 100.0 * kept_stats['deleted_frames_kept'] / kept_stats['total_frames_kept']
    animal_summary = animal_counts.merge(
        kept_stats[['animal_id', 'mean_deleted_pct_kept', 'weighted_deleted_pct_kept']],
        on='animal_id',
        how='left',
    )
else:
    animal_summary = animal_counts.copy()
    animal_summary['mean_deleted_pct_kept'] = np.nan
    animal_summary['weighted_deleted_pct_kept'] = np.nan

display(animal_summary.sort_values('animal_id'))


=== Overall ===
Total valid cells: 48
Kept cells (good SNR >= 5.0 min): 40 (83.3%)
Kept cells - deleted time (% of recording): mean=4.48%, median=0.00%, weighted=5.05%


,animal_id,total_cells,kept_cells,kept_pct,mean_deleted_pct_kept,weighted_deleted_pct_kept
0,CKII_pAce21_PR_20250806,9,9,100.000000,0.000000,0.000000
1,CKII_pAce38_PX_20251126,9,8,88.888889,0.000000,0.000000
2,CKII_pAce45_PX_20260118,16,14,87.500000,5.649526,5.649526
3,CKII_pAce46_PR_20260222,9,5,55.555556,19.577165,19.577165
4,CKII_pAce47_PX_20260128,5,4,80.000000,0.548407,0.548407


In [6]:
# Save tables
out_dir = figures_root / 'CKII_pooled'
out_dir.mkdir(parents=True, exist_ok=True)

cell_csv = out_dir / 'snr_time_varying_cell_stats.csv'
summary_csv = out_dir / 'snr_time_varying_animal_summary.csv'

cell_stats.to_csv(cell_csv, index=False)
animal_summary.to_csv(summary_csv, index=False)

print(f'Saved: {cell_csv}')
print(f'Saved: {summary_csv}')


Saved: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4_2sessions/figures/CKII_pooled/snr_time_varying_cell_stats.csv
Saved: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4_2sessions/figures/CKII_pooled/snr_time_varying_animal_summary.csv
